# Agente Extrator de Metadados Bibliográficos

Pipeline com **CrewAI** que localiza um PDF em uma pasta, extrai seu texto e gera uma entrada bibliográfica estruturada (JSON) usando LLM.

## 1. Configuração inicial
Suprime avisos para manter a saída limpa.

In [84]:
import warnings

warnings.filterwarnings('ignore')

## 2. Variáveis de ambiente
Carrega a `NVIDIA_API_KEY` do `.env` e a expõe como `NVIDIA_NIM_API_KEY` para o SDK da NVIDIA.

In [85]:
import os
from dotenv import load_dotenv


load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")


## 3. Inicialização do LLM
Configura o modelo **Llama 3.1 70B Instruct** via NVIDIA NIM com temperatura baixa (0.2) para respostas mais deterministicas.

In [ ]:
import os
from crewai import LLM, Agent


llm = LLM(
    model="openrouter/openai/gpt-oss-20b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    temperature=0.2
)


## 4. Ferramenta de leitura de PDF e de salvar arquivo .Bib
Tool `read_pdf_content` customizada com `pypdf` que extrai texto de todas as páginas de um PDF. Retorna erro se o arquivo for vazio ou contiver apenas imagens.

Tool `save_bib_file` 

## 5. Schema bibliográfico (Pydantic)
Define `BibliographyEntry` com campos como `title`, `authors`, `year`, `doi`, etc. Usado como `output_json` para garantir que a LLM retorne dados estruturados.

In [87]:
from typing import Optional, List
from pydantic import BaseModel, Field

class BibliographyEntry(BaseModel):
    """Json schema for a single bibliography entry with relaxed constraints to prevent validation errors."""
    
    entry_type: Optional[str] = Field(default="misc", description="The type of publication (e.g., article, inproceedings, book). Defaults to 'misc'.")
    citation_key: Optional[str] = Field(default="UnknownKey", description="A unique citation key. Defaults to 'UnknownKey'.")
    title: Optional[str] = Field(default="Untitled Paper", description="The exact title of the academic paper. Defaults to 'Untitled Paper'.")
    authors: List[str] = Field(default_factory=list, description="List of all authors in the format 'Lastname, Firstname'. Defaults to empty list.")
    year: Optional[int] = Field(default=None, description="The year of publication. Optional.")
    journal_or_booktitle: Optional[str] = Field(default=None, description="The journal name, conference proceedings, or book title.")
    volume: Optional[str] = Field(default=None, description="The volume number.")
    number: Optional[str] = Field(default=None, description="The issue number.")
    pages: Optional[str] = Field(default=None, description="The page range.")
    publisher: Optional[str] = Field(default=None, description="The publisher or organization.")
    doi: Optional[str] = Field(default=None, description="The DOI (Digital Object Identifier) if available.")

In [88]:
import os
from typing import List, Optional
from crewai.tools import BaseTool
from pydantic import BaseModel
from crewai.tools import (tool, BaseTool)
import pypdf

@tool("Read PDF File")
def read_pdf_content(file_path: str) -> str:
    """Reads and extracts all text content from a PDF file. 
    You must pass the complete absolute file path to this tool."""
    try:
        reader = pypdf.PdfReader(file_path)
        extracted_text = ""
        for page_num, page in enumerate(reader.pages):
            page_text = page.extract_text()
            if page_text:
                extracted_text += f"Page {page_num + 1}:\n{page_text}\n"
        return extracted_text if extracted_text.strip() else "Error: PDF is empty or scanned."
    except Exception as e:
        return f"Error reading PDF: {str(e)}"


In [ ]:
import re
from typing import Optional, List
from pydantic import BaseModel, Field, field_validator

class BibliographyEntry(BaseModel):
    """Schema para entrada de bibliografia com limpeza automática de strings."""
    
    entry_type: Optional[str] = Field(default="misc", description="The type of publication (e.g., article, inproceedings, book).")
    citation_key: Optional[str] = Field(default="UnknownKey", description="A unique citation key.")
    title: Optional[str] = Field(default="Untitled Paper", description="The exact title of the academic paper.")
    authors: List[str] = Field(default_factory=list, description="List of all authors in the format 'Lastname, Firstname'.")
    year: Optional[int] = Field(default=None, description="The year of publication.")
    journal_or_booktitle: Optional[str] = Field(default=None, description="The journal name, conference proceedings, or book title.")
    volume: Optional[str] = Field(default=None, description="The volume number.")
    number: Optional[str] = Field(default=None, description="The issue number.")
    pages: Optional[str] = Field(default=None, description="The page range.")
    publisher: Optional[str] = Field(default=None, description="The publisher or organization.")
    doi: Optional[str] = Field(default=None, description="The DOI if available.")

    @field_validator("title", "journal_or_booktitle", "citation_key", "entry_type", mode="before")
    @classmethod
    def clean_text_fields(cls, value):
        if isinstance(value, str):
            cleaned = value.replace("\n", " ").replace("\r", " ")
            cleaned = re.sub(r"\s+", " ", cleaned)
            return cleaned.strip()
        return value

    @field_validator("authors", mode="before")
    @classmethod
    def clean_authors_list(cls, value):
        if isinstance(value, list):
            cleaned_list = []
            for author in value:
                if isinstance(author, str):
                    cleaned_author = author.replace("\n", " ").replace("\r", " ")
                    cleaned_author = re.sub(r"\s+", " ", cleaned_author)
                    cleaned_list.append(cleaned_author.strip())
                else:
                    cleaned_list.append(author)
            return cleaned_list
        return value

## 6. Definição dos agentes
- **Team Archivist**: lista arquivos na pasta `./data`, encontra o melhor match (inclusive com typos) e lê o conteúdo.
- **Academic Metadata Extractor**: recebe o texto bruto e extrai metadados bibliográficos.

In [90]:
from crewai_tools import DirectoryReadTool

from crewai import Agent, Task, Crew, Process

directory_tool = DirectoryReadTool(directory="./data/papers")

agent_archivist = Agent(
    role="Team Archivist",
    goal="Locate the requested PDF document in the folder and extract its raw content using the correct tool.",
    backstory="""You are a strict and literal archivist. You do not possess any personal 
    knowledge about the contents of the files. 
    
    CRITICAL PROTOCOL:
    1. Start by listing all files in the directory using the DirectoryReadTool.
    2. Find the file path that best matches the user's query.
    3. You are FORBIDDEN from guessing, assuming, or making up the content of the file.
    4. You MUST read the actual file content using the "Read PDF File" tool, passing the complete path you found (e.g., '/home/clara/.../agentAILanggraph.pdf').
    5. If you do not execute the "Read PDF File" tool, you fail your mission.""",
    tools=[directory_tool, read_pdf_content],
    llm=llm,
    max_iter=10,
    verbose=True
)


agent_extractor = Agent(
    role="Academic Metadata Extractor",
    goal="Extract all relevant bibliographic entities from the paper to generate a structured bibliography entry.",
    backstory="You are an expert librarian. You take raw text and organize it into clean bibliographic metadata.",
    llm=llm,
    verbose=True
)

save_bibtex_tool = SaveBibTeXTool()

save_bib_tool = SaveBibTeXTool()

agent_creator = Agent(
    role="BibTeX File Creator",
    goal="Take structured academic entries and save them physically on disk as perfectly formatted .bib files.",
    backstory="""You are a precise digital archivist. Your only job is to receive structured 
    academic data, map it to your Save BibTeX Tool, and execute the saving process so that 
    the file is created in the repository.""",
    tools=[save_bib_tool],
    llm=llm,
    max_iter=3,
    verbose=True
)


## 7. Tarefas
- **task_busca**: busca o arquivo no diretório e retorna seu conteúdo bruto.
- **task_extraction**: extrai metadados do texto e retorna JSON conforme `BibliographyEntry`.

In [ ]:
task_busca = Task(
    description="""Search the documents folder for a file that matches or is highly similar to the query '{filename}'.
    First, list all files in the directory. Then, evaluate and identify the single best matching file (even if the query is partial, misspelled, or lacks the extension). 
    Finally, read and return the entire content of that matched file.""",
    expected_output="The raw text content of the best-matched file found in the directory.",
    agent=agent_archivist
)

task_extraction = Task(
    description="Take the raw text provided by the archivist, extract the academic metadata, and structure it.",
    expected_output="A structured JSON object with the paper metadata.",
    agent=agent_extractor,
    output_json=BibliographyEntry
)

task_creation = Task(
    description="""You will receive a structured BibliographyEntry in your context.
    Read EVERY single field present in that entry (especially the clean 'title', 'authors', and 'journal_or_booktitle').
    Call the 'Save BibTeX Tool' mapping each field EXACTLY as received. 
    Do not omit, modify, or leave the title blank.""",
    expected_output="The success message from the Save BibTeX Tool.",
    agent=agent_creator,
    context=[task_extraction]
)

## 8. Montagem da Crew
Cria o `Crew` com os dois agentes e duas tarefas em execução **sequencial** (busca → extração).

In [92]:
crew = Crew(
    agents=[agent_archivist, agent_extractor, agent_creator],
    tasks=[task_busca, task_extraction, task_creation],
    process=Process.sequential, 
    verbose=True
)


## 9. Execução
Dispara o pipeline completo: o Archivist encontra e lê o PDF, depois o Extractor gera a entrada bibliográfica estruturada.

In [93]:

resultado = await crew.kickoff_async(inputs={"filename": "langgraph paper"})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 269d725d-f1eb-45e6-a57a-10c726c7971a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the documents folder for a file that matches or is highly similar to the query 'langgraph         │
│  paper'.                                                                                                        │
│      First, list all files in the directory. Then, evaluate and identify the single best matching file (even    │
│  if the query is partial, misspelled, or lacks the extension).                                                  │
│      Finally, read and return the entire content of that matched file.                                          │
│  ID: 9d06d03b-9801-4df2-9b3f-50f439e8c12d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Team Archivist                                                                                          │
│                                                                                                                 │
│  Task: Search the documents folder for a file that matches or is highly similar to the query 'langgraph         │
│  paper'.                                                                                                        │
│      First, list all files in the directory. Then, evaluate and identify the single best matching file (even    │
│  if the query is partial, misspelled, or lacks the extension).                                                  │
│      Finally, read and return the entire content of that matched file.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#8) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: list_files_in_directory                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool list_files_in_directory executed with result: File paths: 
-/home/clara/Documentos/AgenteExtrator/data/papers/hierarchicalMultiAgent.pdf
- /home/clara/Documentos/AgenteExtrator/data/papers/agentAILanggraph.pdf
- /home/clara/Documentos/AgenteExtra...


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: BibTeX File Creator                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Success: BibTeX citation successfully formatted and saved to 'data\/references\/WangDuan2024AgentAI.bib'       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Take the structured bibliography entry.                                                                  │
│      1. Decide a perfect filename for it (e.g., 'AuthorYear.bib').                                              │
│      2. Format the entry as a valid BibTeX string.                                                              │
│      3. Call the 'Save BibTeX Tool' to create and save the file in 'data/references/'.                          │
│  Agent: BibTeX File Creator                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 269d725d-f1eb-45e6-a57a-10c726c7971a                                                                       │
│  Final Output: Success: BibTeX citation successfully formatted and saved to                                     │
│  'data\/references\/WangDuan2024AgentAI.bib'                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯